# Overhead Lines and Underground Cables

All overhead and underground AC lines are represented as *ACLineSegment* objects. Like all *ConductingEquipment*, all lines and cables are defined with two Terminal objects with *ACDCTerminal:sequenceNumber* set to 1 and 2 to represent the two ends of the line, rather than specifying a from-bus and to-bus. There are four different ways to specify *ACLineSegment* impedances and admittances. The first two use positive and zero sequence values; the third specifies the lower triangular R, X, and B values for each conductor phase; the fourth uses the conductor material, geometry, and spacing. In all cases, the *Conductor:length* attribute is required. A combination of all four methods may be used in a single model to define the network. 

The first way, depicted in the figure below, is to specify the individual positive sequence and zero sequence R, X, and B values as *ACLineSegment* attributes, in a manner similar to the method used to define line impedance in many bus-branch transmission analysis tools, such as PSSE. A power system application importing the CIM model would directly use the specified attributes to run a power flow solution. The *PerLengthImpedance* attribute is left as null. The classes and attributes used in this method are shown in the figure below.

In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [2]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor])
Mermaid(diagram_text)

The second way, displayed in the figure below, is to specify the positive and zero sequence impedance and admittance values on a per-unit-length basis as attributes of the *PerLengthSequenceImpedance* class. A power system application importing the CIM model would multiply the specified R, X, and B values by the *Conductor:length* to determine the overall line impedance. When using this method, all the *ACLineSegment* attributes should be null.

In [6]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance])
Mermaid(diagram_text)

The third way to specify line parameters, shown in the figure below, is to define R, X, and B values for each conductor phase. This method is more useful for distribution networks for which an unbalanced power flow solution is needed. The impedance and admittance values are specified as attributes of the *PhaseImpedanceData* class, which inherits from *PerLengthPhaseImpedance*. Again, all the ACLineSegment attributes are left null. Only conductorCount from 1 to 3 is supported, and there will be 1, 3 or 6 reverse-associated *PhaseImpedanceData* instances that define the lower triangle of the Z and Y matrices per unit length. The row and column attributes must agree with *ACLineSegmentPhase:sequenceNumber*.

In [7]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance,cim.ACLineSegmentPhase,cim.SinglePhaseKind,cim.PhaseImpedanceData,cim.PerLengthPhaseImpedance])
Mermaid(diagram_text)

The fourth way, shown in the figure below, is to specify wire/cable geometry and spacing data instead of impedances. Unlike the previous three methods (where the impedance values were specified through the 61970 Wires package), all attributes of the line are specified through physical attributes as part of the 61968 AssetInfo package.  Conductor spacing is specified by association to *WireSpacingInfo* and *WirePosition*. Conductor geometry is specified through the attributes of *WireInfo*. Cables are specified through the CableInfo class and associated *ConcentricNeutralCableInfo* and *TapeShieldCableInfo* classes.

If there are *ACLineSegmentPhase* instances reverse-associated to the *ACLineSegment*, then per-phase modeling applies. There are several use cases for the *ACLineSegmentPhase* class:
1)	single-phase, two-phase, or three-phase unbalanced primary lines
2)	low-voltage secondary lines using phases s1 and s2
3)	associated WireInfo data where the WireSpacingInfo association exists
4)	assign specific phases to the matrix rows and columns in PerLengthPhaseImpedance. 


It is the application’s responsibility to propagate phasing through terminals to other components, and to identify any miswiring. It is also the application’s responsibility to calculate the impedance and admittance values for the electrical network from the conductor geometry and spacing. The associations between all the AssetInfo classes described above are shown in the figure below.

In [3]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.Conductor,cim.PerLengthImpedance,cim.PerLengthSequenceImpedance,cim.ACLineSegmentPhase,cim.SinglePhaseKind,cim.PhaseImpedanceData,cim.PerLengthPhaseImpedance,cim.PerLengthLineParameter,cim.WireAssemblyInfo,cim.WireSpacingInfo,cim.WireInfo,cim.ConcentricNeutralCableInfo,cim.TapeShieldCableInfo,cim.CableInfo,cim.WireMaterialKind,cim.CableConstructionKind,cim.CableShieldMaterialKind,cim.WireInsulationKind,cim.CableOuterJacketKind])
Mermaid(diagram_text)

----

Some examples related to overhead lines and underground cables are discussed below.


In [5]:
from cimgraph.models import FeederModel
from cimgraph.databases import ConnectionParameters, XMLFile
import cimgraph.data_profile.cimhub_2023 as cim

In [6]:
params = ConnectionParameters(filename='../sample_models/ieee13.xml',
                              cim_profile='cimhub_2023',
                              iec61970_301=8) # file path
file = XMLFile(params) # file read connection
network = FeederModel(container=cim.Feeder(),connection=file) # create feeder model

Example 1: What are the phases of Line named 632645?


In [7]:
result = []
name = '632645'

# Check that the graph contains ACLineSegment
if cim.ACLineSegment in network.graph:
    #Traverse through all ACLineSegment instances in the network graph
    for line in network.graph[cim.ACLineSegment].values():
        # Check if the line's name matches the specified name '632645'
        if name in line.name:
            # Loop through all ACLineSegmentPhases associated with the ACLineSegment
            for phases in line.ACLineSegmentPhases:
                # Append the phase to the result list
                result.append(str(phases.phase))
            break # break after we find line with correct name

# Output the result which contains the phases of the line named '632645'
print(result)

['SinglePhaseKind.C', 'SinglePhaseKind.B']


In [8]:
diagram_text = utils.get_mermaid_path(line,['ACLineSegmentPhases','[0]','phase'])
diagram_text = utils.add_mermaid_path(line,'ACLineSegmentPhases[1].phase', diagram_text)
Mermaid(diagram_text)

Example 2: Find the length of the line named 632645?

In [9]:
results = []
name = '632645'

# Initialize the result variable to store the length of the line
result = None

# Check that the graph contains ACLineSegment
if cim.ACLineSegment in network.graph:
    # Traverse through all ACLineSegment instances in the network graph
    for line in network.graph[cim.ACLineSegment].values():
        # Check if the line's name matches the specified name '632645'
        if name in line.name:
            # Get the length of the ACLineSegment
            results = line.length
            break

# Output the result which contains the length of the line named '632645'
print(results)

152.4


In [10]:
diagram_text = utils.get_mermaid_path(line,'length')
Mermaid(diagram_text)

Example 3: What is the nominal voltage of the line named 632645?

In [11]:
results = []
name = '632645'

# Check that the graph contains ACLineSegment
if cim.ACLineSegment in network.graph:
    # Iterate through all ACLineSegment instances in the network graph
    for line in network.graph[cim.ACLineSegment].values():
        # Check if the line's name contains the target name
        if name in line.name:
            # Get the BaseVoltage object
            base_voltage = line.BaseVoltage
            # Get the nominalVoltage attribute:
            results.append(base_voltage.nominalVoltage)
            break  # Exit loop since we found the required line

print(results)

[4160.0]


In [12]:
diagram_text = utils.get_mermaid_path(line,'BaseVoltage')
diagram_text = utils.add_mermaid_path(base_voltage, 'nominalVoltage', diagram_text)
Mermaid(diagram_text)

In [13]:
# diagram_text = utils.get_mermaid_path(cim.ACLineSegment,'BaseVoltage')
# Mermaid(diagram_text)